In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
import librosa
import numpy as np
import torch.nn.functional as F
from torchvision.models import efficientnet_b0
import torch.nn as nn

# -----------------------------
# Load Model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

model.load_state_dict(torch.load("/content/drive/MyDrive/DeepfakeAudio/models/best_model_v2.pth", map_location=device))
model = model.to(device)
model.eval()

# -----------------------------
# Audio → Mel Spectrogram
# -----------------------------
def preprocess_audio(file_path):
    y, sr = librosa.load(file_path, sr=16000, mono=True)

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Normalize
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    # Resize to CNN input (224x224)
    mel_tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    mel_tensor = F.interpolate(
        mel_tensor,
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    )

    # Convert to 3-channel
    mel_tensor = mel_tensor.repeat(1, 3, 1, 1)

    return mel_tensor


# -----------------------------
# Prediction
# -----------------------------
def predict(file_path):
    x = preprocess_audio(file_path).to(device)

    with torch.no_grad():
        output = model(x)
        probs = F.softmax(output, dim=1)

        pred = torch.argmax(probs, dim=1).item()
        confidence = torch.max(probs).item()

    label = "FAKE" if pred == 0 else "REAL"

    print(f"\nPrediction: {label}")
    print(f"Confidence: {confidence:.4f}")


if __name__ == "__main__":
    path = input("Enter audio file path: ")
    predict(path)

Enter audio file path: /content/drive/MyDrive/DeepfakeAudio/sample.wav

Prediction: REAL
Confidence: 0.9997
